In [276]:
#Data Science Final Project - Abdulsamad Sulyman (AKS2288)
#This notebook is dedicated to data analysis (EDA) of our cleaned 
#CSV weather datasets, including distributions, trends, and seasonal patterns
#IMPORTS
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
from adjustText import adjust_text  

In [190]:
cd /Users/samadsulyman/Desktop/Data Science Project/Processed Data

/Users/samadsulyman/Desktop/Data Science Project/Processed Data


In [191]:
df1 = pd.read_csv("/Users/samadsulyman/Desktop/Data Science Project/Processed Data/ProcessedDataset1.csv")
print(df1.columns.tolist())
df2 = pd.read_csv("/Users/samadsulyman/Desktop/Data Science Project/Processed Data/ProcessedDataset2.csv")
print(df2.columns.tolist())

['date', 'AverageTemperature', 'AverageTemperatureUncertainty', 'City', 'Country', 'Latitude', 'Longitude', 'City_match']
['date', 'Region', 'Country', 'State', 'City', 'AvgTemperature', 'City_match']


In [236]:
# =============================
# Setup font to Times New Roman
# =============================
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif']
plt.rcParams['font.size'] = 11

# =============================
# Setup output directory
# =============================
output_dir = "/Users/samadsulyman/Desktop/Data Science Project/Plots - Climate"
os.makedirs(output_dir, exist_ok=True)
print(f"Saving plots to: {output_dir}")

# =============================
# Load Data
# =============================
df1 = pd.read_csv("/Users/samadsulyman/Desktop/Data Science Project/Processed Data/ProcessedDataset1.csv")
df2 = pd.read_csv("/Users/samadsulyman/Desktop/Data Science Project/Processed Data/ProcessedDataset2.csv")

df1["date"] = pd.to_datetime(df1["date"], errors="coerce")
df2["date"] = pd.to_datetime(df2["date"], errors="coerce")

df1["Temp_F"] = df1["AverageTemperature"] * 9/5 + 32


# =============================
# Filtering function
# =============================
def filter_insufficient_seasonal_coverage(df, date_col, temp_col):
    df = df.copy()
    df.loc[:, "month"] = df[date_col].dt.month
    df.loc[:, "year"] = df[date_col].dt.year

    df.loc[:, "season"] = pd.cut(
        df["month"],
        bins=[0, 2, 5, 8, 11, 12],
        labels=["Winter", "Spring", "Summer", "Fall", "Winter_adj"],
        right=True
    )
    df["season"] = df["season"].astype(str).replace("Winter_adj", "Winter")

    seasonal_counts = df.groupby(["City", "year", "season"]).size().reset_index(name="count")
    valid_years = seasonal_counts.groupby(["City", "year"])["count"].apply(lambda x: (x >= 2).all())
    valid_years = valid_years[valid_years].reset_index()[["City", "year"]]

    return df.merge(valid_years, on=["City", "year"], how="inner")


# =============================
# DATASET 1 (1780–2010)
# =============================
print("\n" + "="*60)
print("Processing Dataset 1 (1780-2010)...")
print("="*60)

d1 = df1[(df1["date"].dt.year >= 1780) & (df1["date"].dt.year <= 2010)]
d1 = filter_insufficient_seasonal_coverage(d1, "date", "Temp_F")

# Calculate 6-month rolling averages
d1_m = (
    d1.groupby([d1["City"], d1["date"].dt.to_period("M")])["Temp_F"].mean().reset_index()
)
d1_m["date"] = d1_m["date"].dt.to_timestamp()

# Store results for summary bar chart
dataset1_trends = []

for city in d1_m["City"].unique():
    print(f"  Processing {city}...")
    c = d1_m[d1_m["City"] == city].sort_values("date").copy()
    
    # Calculate 6-month rolling average
    c["roll_6m"] = c["Temp_F"].rolling(window=6, center=False).mean()
    
    # Drop NaN values from rolling average
    c_plot = c.dropna(subset=["roll_6m"])

    plt.figure(figsize=(12, 6))
    
    # Plot the 6-month rolling average line
    plt.plot(c_plot["date"], c_plot["roll_6m"], linewidth=2, color="dodgerblue", 
             label="6-Month Rolling Average")

    # Calculate and plot linear trend using the rolling average data
    x = c_plot["date"].map(pd.Timestamp.toordinal).values
    y = c_plot["roll_6m"].values
    coeffs = np.polyfit(x, y, 1)
    m = coeffs[0]
    b = coeffs[1]
    linear_fit = m * x + b
    
    plt.plot(c_plot["date"], linear_fit, linewidth=3.5, color="red", alpha=0.85, 
             linestyle='--', label="Linear Trend", zorder=10)

    plt.title(f"Dataset 1 — {city} Climate Trend (1780–2010)", fontsize=16, weight='normal')
    plt.xlabel("Year", fontsize=13)
    plt.ylabel("Temperature (°F)", fontsize=13)
    plt.legend(loc='best', fontsize=11, frameon=True, fancybox=False, edgecolor='black')
    plt.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
    plt.tight_layout()
    
    # Save plot
    filename = f"Dataset1_{city.replace(' ', '_')}_1780-2010.png"
    filepath = os.path.join(output_dir, filename)
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()
    
    # Calculate statistics
    temp_change = linear_fit[-1] - linear_fit[0]
    years = (c_plot["date"].iloc[-1] - c_plot["date"].iloc[0]).days / 365.25
    change_per_decade = (temp_change / years) * 10
    
    dataset1_trends.append({"City": city, "Change_per_Decade": change_per_decade})



# =============================
# DATASET 2 (1995–2010)
# =============================
print("\n" + "="*60)
print("Processing Dataset 2 (1995-2010)...")
print("="*60)

d2 = df2[df2["date"].dt.year >= 1995]
d2 = filter_insufficient_seasonal_coverage(d2, "date", "AvgTemperature")

# Calculate 6-month rolling averages
d2_m = (
    d2.groupby([d2["City"], d2["date"].dt.to_period("M")])["AvgTemperature"].mean().reset_index()
)
d2_m["date"] = d2_m["date"].dt.to_timestamp()

# Store results for summary bar chart
dataset2_trends = []

for city in d2_m["City"].unique():
    print(f"  Processing {city}...")
    c = d2_m[d2_m["City"] == city].sort_values("date").copy()
    
    # Calculate 6-month rolling average
    c["roll_6m"] = c["AvgTemperature"].rolling(window=6, center=False).mean()
    
    # Drop NaN values from rolling average
    c_plot = c.dropna(subset=["roll_6m"])

    plt.figure(figsize=(12, 6))
    
    # Plot the 6-month rolling average line
    plt.plot(c_plot["date"], c_plot["roll_6m"], linewidth=2, color="seagreen",
             label="6-Month Rolling Average")

    # Calculate and plot linear trend using the rolling average data
    x = c_plot["date"].map(pd.Timestamp.toordinal).values
    y = c_plot["roll_6m"].values
    coeffs = np.polyfit(x, y, 1)
    m = coeffs[0]
    b = coeffs[1]
    linear_fit = m * x + b
    
    plt.plot(c_plot["date"], linear_fit, linewidth=3.5, color="red", alpha=0.85,
             linestyle='--', label="Linear Trend", zorder=10)

    plt.xlim(pd.Timestamp("1995-01-01"), c_plot["date"].max())
    plt.title(f"Dataset 2 — {city} Climate Trend (1995–2010)", fontsize=16, weight='normal')
    plt.xlabel("Year", fontsize=13)
    plt.ylabel("Temperature (°F)", fontsize=13)
    plt.legend(loc='best', fontsize=11, frameon=True, fancybox=False, edgecolor='black')
    plt.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
    plt.tight_layout()
    
    # Save plot
    filename = f"Dataset2_{city.replace(' ', '_')}_1995-2010.png"
    filepath = os.path.join(output_dir, filename)
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()
    
    # Calculate statistics
    temp_change = linear_fit[-1] - linear_fit[0]
    years = (c_plot["date"].iloc[-1] - c_plot["date"].iloc[0]).days / 365.25
    change_per_decade = (temp_change / years) * 10
    
    dataset2_trends.append({"City": city, "Change_per_Decade": change_per_decade})


# =============================
# SUMMARY BAR CHARTS
# =============================
print("\n" + "="*60)
print("Creating summary bar charts...")
print("="*60)

# Dataset 1 Bar Chart
df1_trends = pd.DataFrame(dataset1_trends)
plt.figure(figsize=(12, 6))
bars1 = plt.bar(df1_trends["City"], df1_trends["Change_per_Decade"], 
                color="dodgerblue", alpha=0.8, edgecolor="navy", linewidth=1.5)

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.3f}°F',
             ha='center', va='bottom', fontsize=10)

plt.title("Dataset 1 — Average Temperature Change per Decade by City (1780–2010)", 
          fontsize=16, weight='normal')
plt.xlabel("City", fontsize=13)
plt.ylabel("Temperature Change (°F/decade)", fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3, axis='y', linestyle='-', linewidth=0.5)
plt.axhline(y=0, color='black', linewidth=1, linestyle='-')
plt.tight_layout()

# Save bar chart
filepath = os.path.join(output_dir, "Dataset1_Summary_ChangePerDecade.png")
plt.savefig(filepath, dpi=300, bbox_inches='tight')
plt.close()
print("  Saved Dataset 1 summary bar chart")


# Dataset 2 Bar Chart
df2_trends = pd.DataFrame(dataset2_trends)
plt.figure(figsize=(12, 6))
bars2 = plt.bar(df2_trends["City"], df2_trends["Change_per_Decade"], 
                color="seagreen", alpha=0.8, edgecolor="darkgreen", linewidth=1.5)

# Add value labels on bars
for bar in bars2:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.3f}°F',
             ha='center', va='bottom', fontsize=10)

plt.title("Dataset 2 — Average Temperature Change per Decade by City (1995–2010)", 
          fontsize=16, weight='normal')
plt.xlabel("City", fontsize=13)
plt.ylabel("Temperature Change (°F/decade)", fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3, axis='y', linestyle='-', linewidth=0.5)
plt.axhline(y=0, color='black', linewidth=1, linestyle='-')
plt.tight_layout()

# Save bar chart
filepath = os.path.join(output_dir, "Dataset2_Summary_ChangePerDecade.png")
plt.savefig(filepath, dpi=300, bbox_inches='tight')
plt.close()
print("  Saved Dataset 2 summary bar chart")

print("\n" + "="*60)
print("SUMMARY: Temperature Change per Decade")
print("="*60)
print("\nDataset 1 (1780-2010):")
print(df1_trends.to_string(index=False))
print("\nDataset 2 (1995-2010):")
print(df2_trends.to_string(index=False))

print("\n" + "="*60)
print("ALL PLOTS SAVED SUCCESSFULLY!")
print("="*60)
print(f"\nLocation: {output_dir}")
print(f"Total plots saved: {len(dataset1_trends) + len(dataset2_trends) + 2}")
print(f"  - Dataset 1 city plots: {len(dataset1_trends)}")
print(f"  - Dataset 2 city plots: {len(dataset2_trends)}")
print(f"  - Summary bar charts: 2")

Saving plots to: /Users/samadsulyman/Desktop/Data Science Project/Plots - Climate

Processing Dataset 1 (1780-2010)...
  Processing Abidjan...
  Processing Accra...
  Processing Addis Abeba...
  Processing Algiers...
  Processing Cairo...
  Processing Dakar...
  Processing Dar Es Salaam...
  Processing Johannesburg...
  Processing Kamalia...
  Processing Kampala...
  Processing Khartoum...
  Processing Kinshasa...
  Processing Lagos...
  Processing Luanda...
  Processing Maputo...
  Processing Nairobi...

Processing Dataset 2 (1995-2010)...
  Processing Abidjan...
  Processing Addis Ababa...
  Processing Algiers...
  Processing Cairo...
  Processing Dakar...
  Processing Dar Es Salaam...
  Processing Kampala...
  Processing Lagos...
  Processing Maputo...
  Processing Nairobi...

Creating summary bar charts...
  Saved Dataset 1 summary bar chart
  Saved Dataset 2 summary bar chart

SUMMARY: Temperature Change per Decade

Dataset 1 (1780-2010):
         City  Change_per_Decade
      Abi

In [244]:
# =============================
# Setup font to Times New Roman
# =============================
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif']
plt.rcParams['font.size'] = 11

# =============================
# Setup output directory
# =============================
output_dir = "/Users/samadsulyman/Desktop/Data Science Project/Plots - Crop Yield"
os.makedirs(output_dir, exist_ok=True)
print(f"Saving plots to: {output_dir}")

# =============================
# Load Data
# =============================
print("\n" + "="*60)
print("Loading Dataset 3 (Crop Production Data)...")
print("="*60)

df3 = pd.read_csv("/Users/samadsulyman/Desktop/Data Science Project/Processed Data/ProcessedDataset3.csv")

print(f"Dataset shape: {df3.shape}")
print(f"\nAvailable columns: {df3.columns.tolist()}")
print(f"\nUnique Areas (Cities): {df3['Area'].nunique()}")
print(f"Year range: {df3['Year'].min()} - {df3['Year'].max()}")
print(f"\nUnique Items (Crops): {df3['Item'].nunique()}")

# =============================
# Filter for Export Quantity data
# =============================
print("\n" + "="*60)
print("Filtering for Export Quantity data...")
print("="*60)

# Look for export-related elements
print(f"Unique Elements: {df3['Element'].unique()}")

# Filter for export quantity data
export_data = df3[df3['Element'].str.contains('Export', case=False, na=False)].copy()

if len(export_data) == 0:
    print("No export data found. Using production/yield data instead...")
    export_data = df3[df3['Element'].str.contains('Production|Yield', case=False, na=False)].copy()

print(f"Filtered data shape: {export_data.shape}")

# Remove rows with missing values
export_data = export_data.dropna(subset=['Value', 'Year', 'Area'])

# =============================
# Calculate Expected Growth for African Cities
# =============================
def calculate_expected_growth(df_city):
    """
    Calculate expected growth based on 3.5% annual growth rate for African cities.
    This reflects higher population growth rates in Africa.
    """
    df_city = df_city.sort_values('Year').copy()
    years = df_city['Year'].values
    
    if len(years) < 2:
        return df_city
    
    # Calculate cumulative growth with 3.5% annual rate
    base_year = years[0]
    df_city['Years_Since_Base'] = df_city['Year'] - base_year
    df_city['Expected_Growth_Factor'] = (1.035) ** df_city['Years_Since_Base']
    
    return df_city


# =============================
# Analyze Crop Yield vs Expected Growth by City
# =============================
def analyze_crop_yield_by_city(df, output_dir):
    """
    Analyze crop yield vs expected growth for each city.
    Returns summary statistics for all cities.
    """
    
    results = []
    
    # Get cities that match our climate analysis
    cities = df['Area'].unique()
    
    print(f"\nAnalyzing {len(cities)} cities...")
    
    for city in cities:
        print(f"\n  Processing {city}...")
        
        # Filter data for this city
        city_data = df[df['Area'] == city].copy()
        
        if len(city_data) < 5:  # Need at least 5 data points
            print(f"    Skipping {city} - insufficient data")
            continue
        
        # Group by year and sum all crop exports/production
        yearly_data = city_data.groupby('Year')['Value'].sum().reset_index()
        yearly_data['Area'] = city
        
        if len(yearly_data) < 5:
            continue
        
        # Calculate expected growth (4% for African cities)
        yearly_data = calculate_expected_growth(yearly_data)
        
        # Calculate baseline (first year value)
        baseline_value = yearly_data['Value'].iloc[0]
        
        # Calculate expected value based on growth factor
        yearly_data['Expected_Value'] = baseline_value * yearly_data['Expected_Growth_Factor']
        
        # Calculate actual growth factor
        yearly_data['Actual_Growth_Factor'] = yearly_data['Value'] / baseline_value
        
        # Calculate difference (Actual vs Expected)
        yearly_data['Growth_Difference'] = yearly_data['Actual_Growth_Factor'] - yearly_data['Expected_Growth_Factor']
        yearly_data['Percent_Difference'] = (yearly_data['Growth_Difference'] / yearly_data['Expected_Growth_Factor']) * 100
        
        # Create visualization
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
        
        # Plot 1: Actual vs Expected Values
        ax1.plot(yearly_data['Year'], yearly_data['Value'], 
                linewidth=2.5, color='darkgreen', marker='o', markersize=5,
                label='Actual Production/Export')
        ax1.plot(yearly_data['Year'], yearly_data['Expected_Value'], 
                linewidth=2.5, color='red', linestyle='--', marker='s', markersize=5,
                label='Expected (3.5% annual growth)')
        ax1.fill_between(yearly_data['Year'], yearly_data['Value'], yearly_data['Expected_Value'],
                        alpha=0.2, color='gray')
        
        ax1.set_title(f"{city} — Actual vs Expected Crop Production", fontsize=16, weight='normal')
        ax1.set_xlabel("Year", fontsize=13)
        ax1.set_ylabel("Total Production/Export Value", fontsize=13)
        ax1.legend(loc='best', fontsize=11, frameon=True, fancybox=False, edgecolor='black')
        ax1.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
        
        # Plot 2: Percentage Difference
        colors = ['green' if x >= 0 else 'red' for x in yearly_data['Percent_Difference']]
        ax2.bar(yearly_data['Year'], yearly_data['Percent_Difference'], 
                color=colors, alpha=0.7, edgecolor='black', linewidth=1)
        ax2.axhline(y=0, color='black', linewidth=1.5, linestyle='-')
        
        ax2.set_title(f"{city} — Production Deviation from Expected Growth", fontsize=16, weight='normal')
        ax2.set_xlabel("Year", fontsize=13)
        ax2.set_ylabel("Deviation from Expected (%)", fontsize=13)
        ax2.grid(True, alpha=0.3, axis='y', linestyle='-', linewidth=0.5)
        
        plt.tight_layout()
        
        # Save plot
        filename = f"CropYield_{city.replace(' ', '_')}_Analysis.png"
        filepath = os.path.join(output_dir, filename)
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        plt.close()
        
        # Calculate summary statistics
        final_year_diff = yearly_data['Percent_Difference'].iloc[-1]
        avg_diff = yearly_data['Percent_Difference'].mean()
        
        results.append({
            'City': city,
            'Years_Analyzed': f"{yearly_data['Year'].min()}-{yearly_data['Year'].max()}",
            'Final_Year_Deviation_%': final_year_diff,
            'Average_Deviation_%': avg_diff,
            'Data_Points': len(yearly_data)
        })
        
        print(f"    Final year deviation: {final_year_diff:.2f}%")
        print(f"    Average deviation: {avg_diff:.2f}%")
    
    return pd.DataFrame(results)

# =============================
# Run Analysis
# =============================
print("\n" + "="*60)
print("Running Crop Yield Analysis (3.5% Growth Rate)...")
print("="*60)

results_df = analyze_crop_yield_by_city(export_data, output_dir)

# =============================
# Create Summary Bar Chart
# =============================
if len(results_df) > 0:
    print("\n" + "="*60)
    print("Creating summary visualization...")
    print("="*60)
    
    # Sort by average deviation
    results_df = results_df.sort_values('Average_Deviation_%', ascending=False)
    
    plt.figure(figsize=(14, 8))
    colors = ['green' if x >= 0 else 'red' for x in results_df['Average_Deviation_%']]
    bars = plt.bar(results_df['City'], results_df['Average_Deviation_%'], 
                   color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%',
                ha='center', va='bottom' if height >= 0 else 'top', fontsize=9)
    
    plt.axhline(y=0, color='black', linewidth=2, linestyle='-')
    plt.title("Average Crop Production Deviation from Expected Growth by City", 
              fontsize=16, weight='normal')
    plt.xlabel("City", fontsize=13)
    plt.ylabel("Average Deviation from Expected (%)", fontsize=13)
    plt.xticks(rotation=45, ha='right')
    plt.grid(True, alpha=0.3, axis='y', linestyle='-', linewidth=0.5)
    plt.tight_layout()
    
    filepath = os.path.join(output_dir, "Summary_CropYield_Deviation.png")
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()
    print("  Saved summary bar chart")
    
    # Print summary table
    print("\n" + "="*60)
    print("SUMMARY: Crop Yield vs Expected Growth")
    print("="*60)
    print(results_df.to_string(index=False))
    
    print("\n" + "="*60)
    print("INTERPRETATION:")
    print("="*60)
    print("Positive values: Production EXCEEDED expected growth (good productivity)")
    print("Negative values: Production BELOW expected growth (concerning trend)")
    print("\nExpected growth assumes 3.5% annual population growth rate (typical for African cities)")
    print("Deviations may indicate:")
    print("  - Climate change impacts on agriculture")
    print("  - Technological improvements (positive)")
    print("  - Resource depletion or degradation (negative)")
    print("  - Policy changes affecting agricultural output")
    
    print("\n" + "="*60)
    print("ALL PLOTS SAVED SUCCESSFULLY!")
    print("="*60)
    print(f"\nLocation: {output_dir}")
    print(f"Total plots saved: {len(results_df) + 1}")
else:
    print("\nNo results to display. Check data filtering criteria.")

Saving plots to: /Users/samadsulyman/Desktop/Data Science Project/Plots - Crop Yield

Loading Dataset 3 (Crop Production Data)...
Dataset shape: (292809, 11)

Available columns: ['Year', 'Area Code', 'Area', 'Item Code', 'Item', 'Element Code', 'Element', 'Unit', 'Value', 'Flag', 'Area_match']

Unique Areas (Cities): 22
Year range: 1961 - 2019

Unique Items (Crops): 161

Filtering for Export Quantity data...
Unique Elements: ['Area harvested' 'Yield' 'Production']
No export data found. Using production/yield data instead...
Filtered data shape: (193193, 11)

Running Crop Yield Analysis (3.5% Growth Rate)...

Analyzing 22 cities...

  Processing Algeria...
    Final year deviation: -24.60%
    Average deviation: -35.07%

  Processing Angola...
    Final year deviation: -29.37%
    Average deviation: -35.81%

  Processing C��te d'Ivoire...


/var/folders/26/2f2vy6w539b6382kp83vyy_w0000gn/T/ipykernel_30793/4262148071.py:158: UserWarning: Glyph 65533 (\N{REPLACEMENT CHARACTER}) missing from current font.
  plt.tight_layout()
/var/folders/26/2f2vy6w539b6382kp83vyy_w0000gn/T/ipykernel_30793/4262148071.py:163: UserWarning: Glyph 65533 (\N{REPLACEMENT CHARACTER}) missing from current font.
  plt.savefig(filepath, dpi=300, bbox_inches='tight')


    Final year deviation: -2.60%
    Average deviation: 15.02%

  Processing Democratic Republic of the Congo...
    Final year deviation: -49.46%
    Average deviation: -36.55%

  Processing Egypt...
    Final year deviation: -32.86%
    Average deviation: -8.45%

  Processing Ethiopia...
    Final year deviation: 59.46%
    Average deviation: 30.41%

  Processing Ghana...
    Final year deviation: 13.87%
    Average deviation: -10.39%

  Processing Kenya...
    Final year deviation: -33.78%
    Average deviation: -2.31%

  Processing Mozambique...
    Final year deviation: -67.98%
    Average deviation: -45.58%

  Processing Nigeria...
    Final year deviation: -27.29%
    Average deviation: -20.29%

  Processing Senegal...
    Final year deviation: -39.78%
    Average deviation: -34.03%

  Processing South Africa...
    Final year deviation: -66.57%
    Average deviation: -26.64%

  Processing Sudan...
    Final year deviation: 7.48%
    Average deviation: 13.02%

  Processing Ugand

/var/folders/26/2f2vy6w539b6382kp83vyy_w0000gn/T/ipykernel_30793/4262148071.py:223: UserWarning: Glyph 65533 (\N{REPLACEMENT CHARACTER}) missing from current font.
  plt.tight_layout()
/var/folders/26/2f2vy6w539b6382kp83vyy_w0000gn/T/ipykernel_30793/4262148071.py:226: UserWarning: Glyph 65533 (\N{REPLACEMENT CHARACTER}) missing from current font.
  plt.savefig(filepath, dpi=300, bbox_inches='tight')


  Saved summary bar chart

SUMMARY: Crop Yield vs Expected Growth
                            City Years_Analyzed  Final_Year_Deviation_%  Average_Deviation_%  Data_Points
                        Ethiopia      1993-2019               59.462131            30.410122           27
                  C��te d'Ivoire      1961-2019               -2.600999            15.016942           59
                           Sudan      2012-2019                7.478546            13.022102            8
                 Northern Africa      1961-2019              -26.200947            -2.065997           59
                           Kenya      1961-2019              -33.779655            -2.310105           59
     United Republic of Tanzania      1961-2019              -16.499453            -8.429510           59
                           Egypt      1961-2019              -32.861087            -8.445823           59
                           Ghana      1961-2019               13.870986           -10.

In [256]:
# =============================
# Setup font to Times New Roman
# =============================
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif']
plt.rcParams['font.size'] = 11

# =============================
# Setup output directory
# =============================
output_dir = "/Users/samadsulyman/Desktop/Data Science Project/Plots - Correlation"
os.makedirs(output_dir, exist_ok=True)
print(f"Saving correlation plots to: {output_dir}")

# =============================
# Load Climate Data (Datasets 1 & 2)
# =============================
print("\n" + "="*60)
print("Loading Climate Data...")
print("="*60)

df1 = pd.read_csv("/Users/samadsulyman/Desktop/Data Science Project/Processed Data/ProcessedDataset1.csv")
df2 = pd.read_csv("/Users/samadsulyman/Desktop/Data Science Project/Processed Data/ProcessedDataset2.csv")

df1["date"] = pd.to_datetime(df1["date"], errors="coerce")
df2["date"] = pd.to_datetime(df2["date"], errors="coerce")
df1["Temp_F"] = df1["AverageTemperature"] * 9/5 + 32

# =============================
# Load Crop Data (Dataset 3)
# =============================
print("\nLoading Crop Production Data...")
df3 = pd.read_csv("/Users/samadsulyman/Desktop/Data Science Project/Processed Data/ProcessedDataset3.csv")

# Filter for export/production data
export_data = df3[df3['Element'].str.contains('Export|Production', case=False, na=False)].copy()
export_data = export_data.dropna(subset=['Value', 'Year', 'Area'])

# =============================
# Calculate Climate Trends
# =============================
def calculate_climate_trend(df, temp_col, start_year, end_year):
    """Calculate temperature change per decade for a city"""
    df = df[(df["date"].dt.year >= start_year) & (df["date"].dt.year <= end_year)].copy()
    
    if len(df) < 10:
        return None
    
    # Monthly aggregation
    monthly = df.groupby([df["City"], df["date"].dt.to_period("M")])[temp_col].mean().reset_index()
    monthly["date"] = monthly["date"].dt.to_timestamp()
    
    results = {}
    for city in monthly["City"].unique():
        city_data = monthly[monthly["City"] == city].sort_values("date")
        
        # 6-month rolling average
        city_data["roll_6m"] = city_data[temp_col].rolling(window=6, center=False).mean()
        city_data = city_data.dropna(subset=["roll_6m"])
        
        if len(city_data) < 10:
            continue
        
        # Calculate linear trend
        x = city_data["date"].map(pd.Timestamp.toordinal).values
        y = city_data["roll_6m"].values
        
        coeffs = np.polyfit(x, y, 1)
        slope = coeffs[0]
        
        # Convert to change per decade
        years = (city_data["date"].iloc[-1] - city_data["date"].iloc[0]).days / 365.25
        temp_change = slope * (x[-1] - x[0])
        change_per_decade = (temp_change / years) * 10
        
        results[city] = change_per_decade
    
    return results

print("\nCalculating climate trends...")
climate_trends_1 = calculate_climate_trend(df1, "Temp_F", 1780, 2010)
climate_trends_2 = calculate_climate_trend(df2, "AvgTemperature", 1995, 2010)

# Combine climate trends (prefer dataset 2 for overlap period)
climate_trends = {**climate_trends_1, **climate_trends_2}

# =============================
# Calculate Crop Production Separation Rate
# =============================
def calculate_separation_rate(df):
    """Calculate rate of separation between actual and expected crop production"""
    
    results = {}
    
    for city in df['Area'].unique():
        city_data = df[df['Area'] == city].copy()
        
        if len(city_data) < 5:
            continue
        
        # Group by year
        yearly = city_data.groupby('Year')['Value'].sum().reset_index()
        
        if len(yearly) < 5:
            continue
        
        yearly = yearly.sort_values('Year')
        
        # Calculate expected growth (3.5% annual)
        base_year = yearly['Year'].iloc[0]
        yearly['Years_Since_Base'] = yearly['Year'] - base_year
        yearly['Expected_Growth_Factor'] = (1.035) ** yearly['Years_Since_Base']
        
        baseline = yearly['Value'].iloc[0]
        yearly['Expected_Value'] = baseline * yearly['Expected_Growth_Factor']
        yearly['Actual_Growth_Factor'] = yearly['Value'] / baseline
        
        # Calculate separation (Actual - Expected growth factor)
        yearly['Separation'] = yearly['Actual_Growth_Factor'] - yearly['Expected_Growth_Factor']
        
        # Calculate rate of separation change (linear trend of separation over time)
        x = yearly['Year'].values
        y = yearly['Separation'].values
        
        if len(x) >= 3:
            slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
            # Slope represents how fast the gap is widening/narrowing per year
            results[city] = {
                'separation_rate': slope,
                'r_squared': r_value**2,
                'final_separation': y[-1],
                'years': f"{x[0]}-{x[-1]}"
            }
    
    return results

print("\nCalculating crop production separation rates...")
crop_separation = calculate_separation_rate(export_data)

# =============================
# City to Country Mapping
# =============================
city_to_country = {
    # Climate cities -> Crop countries
    'Abidjan': "Côte d'Ivoire",
    'Accra': 'Ghana',
    'Addis Abeba': 'Ethiopia',
    'Addis Ababa': 'Ethiopia',
    'Algiers': 'Algeria',
    'Cairo': 'Egypt',
    'Dakar': 'Senegal',
    'Dar Es Salaam': 'United Republic of Tanzania',
    'Johannesburg': 'South Africa',
    'Kamalia': 'Uganda',
    'Kampala': 'Uganda',
    'Khartoum': 'Sudan',
    'Kinshasa': 'Democratic Republic of the Congo',
    'Lagos': 'Nigeria',
    'Luanda': 'Angola',
    'Maputo': 'Mozambique',
    'Nairobi': 'Kenya'
}

# Fix encoding issues in crop data if they exist
crop_separation_fixed = {}
for country, data in crop_separation.items():
    # Normalize the encoding issue
    if "Cï¿½ï¿½te d'Ivoire" in country:
        country = "Côte d'Ivoire"
    crop_separation_fixed[country] = data

crop_separation = crop_separation_fixed

# =============================
# Merge Climate and Crop Data
# =============================
print("\nMerging datasets...")

correlation_data = []
for city, country in city_to_country.items():
    if city in climate_trends and country in crop_separation:
        correlation_data.append({
            'City': city,
            'Country': country,
            'Climate_Change_per_Decade_F': climate_trends[city],
            'Crop_Separation_Rate': crop_separation[country]['separation_rate'],
            'Final_Separation': crop_separation[country]['final_separation'],
            'R_Squared': crop_separation[country]['r_squared'],
            'Years': crop_separation[country]['years']
        })

corr_df = pd.DataFrame(correlation_data)

if len(corr_df) == 0:
    print("ERROR: No matching cities found between climate and crop data!")
    print(f"Climate cities: {list(climate_trends.keys())}")
    print(f"Crop countries: {list(crop_separation.keys())}")
    
    # Print detailed mapping for debugging
    print("\nAttempted matches:")
    for city, country in city_to_country.items():
        climate_available = city in climate_trends
        crop_available = country in crop_separation
        print(f"  {city} -> {country}: Climate={climate_available}, Crop={crop_available}")
else:
    print(f"\nSuccessfully matched {len(corr_df)} cities for correlation analysis")
    print(corr_df[['City', 'Country']].to_string(index=False))

# =============================
# PLOT 1: Scatter Plot - Climate vs Crop Separation Rate
# =============================
if len(corr_df) > 0:
    plt.figure(figsize=(12, 8))
    
    colors = ['red' if x < 0 else 'green' for x in corr_df['Crop_Separation_Rate']]
    plt.scatter(corr_df['Climate_Change_per_Decade_F'], 
                corr_df['Crop_Separation_Rate'],
                s=200, alpha=0.6, c=colors, edgecolor='black', linewidth=1.5)
    
    # Add city labels
    for idx, row in corr_df.iterrows():
        plt.annotate(row['City'], 
                    (row['Climate_Change_per_Decade_F'], row['Crop_Separation_Rate']),
                    fontsize=9, ha='center', va='bottom')
    
    # Add trend line
    if len(corr_df) >= 3:
        z = np.polyfit(corr_df['Climate_Change_per_Decade_F'], corr_df['Crop_Separation_Rate'], 1)
        p = np.poly1d(z)
        x_trend = np.linspace(corr_df['Climate_Change_per_Decade_F'].min(), 
                             corr_df['Climate_Change_per_Decade_F'].max(), 100)
        plt.plot(x_trend, p(x_trend), "b--", linewidth=2, alpha=0.7, label='Trend Line')
        
        # Calculate correlation
        corr_coef = np.corrcoef(corr_df['Climate_Change_per_Decade_F'], 
                                corr_df['Crop_Separation_Rate'])[0,1]
        plt.text(0.05, 0.95, f'Correlation: {corr_coef:.3f}', 
                transform=plt.gca().transAxes, fontsize=12, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.axhline(y=0, color='black', linewidth=1, linestyle='-', alpha=0.3)
    plt.axvline(x=0, color='black', linewidth=1, linestyle='-', alpha=0.3)
    
    plt.xlabel("Temperature Change per Decade (°F)", fontsize=13)
    plt.ylabel("Crop Production Separation Rate (per year)", fontsize=13)
    plt.title("Climate Change vs Agricultural Productivity Gap", fontsize=16, weight='normal')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    
    filepath = os.path.join(output_dir, "01_Climate_vs_CropSeparation_Scatter.png")
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved Plot 1: Scatter plot")

# =============================
# PLOT 2: Bar Chart - Cities Ranked by Separation Rate
# =============================
if len(corr_df) > 0:
    corr_df_sorted = corr_df.sort_values('Crop_Separation_Rate', ascending=True)
    
    plt.figure(figsize=(12, 8))
    colors = ['red' if x < 0 else 'green' for x in corr_df_sorted['Crop_Separation_Rate']]
    bars = plt.barh(corr_df_sorted['City'], corr_df_sorted['Crop_Separation_Rate'],
                    color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    
    plt.axvline(x=0, color='black', linewidth=2, linestyle='-')
    plt.xlabel("Crop Separation Rate (per year)", fontsize=13)
    plt.ylabel("City", fontsize=13)
    plt.title("Agricultural Productivity Gap: Widening vs Narrowing", fontsize=16, weight='normal')
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    
    filepath = os.path.join(output_dir, "02_CropSeparation_Ranking.png")
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved Plot 2: Separation rate ranking")

# =============================
# PLOT 3: Dual Axis - Climate and Crop Trends by City
# =============================
if len(corr_df) > 0:
    corr_df_sorted = corr_df.sort_values('Climate_Change_per_Decade_F', ascending=False)
    
    fig, ax1 = plt.subplots(figsize=(14, 8))
    
    x_pos = np.arange(len(corr_df_sorted))
    
    # Temperature change
    color1 = 'tab:red'
    ax1.set_xlabel('City', fontsize=13)
    ax1.set_ylabel('Temperature Change (°F/decade)', fontsize=13, color=color1)
    ax1.bar(x_pos - 0.2, corr_df_sorted['Climate_Change_per_Decade_F'], 
            0.4, label='Climate Change', color=color1, alpha=0.7, edgecolor='darkred', linewidth=1.5)
    ax1.tick_params(axis='y', labelcolor=color1)
    ax1.axhline(y=0, color='black', linewidth=1, linestyle='-', alpha=0.3)
    
    # Crop separation rate
    ax2 = ax1.twinx()
    color2 = 'tab:green'
    ax2.set_ylabel('Crop Separation Rate (per year)', fontsize=13, color=color2)
    ax2.bar(x_pos + 0.2, corr_df_sorted['Crop_Separation_Rate'], 
            0.4, label='Crop Gap Rate', color=color2, alpha=0.7, edgecolor='darkgreen', linewidth=1.5)
    ax2.tick_params(axis='y', labelcolor=color2)
    ax2.axhline(y=0, color='black', linewidth=1, linestyle='-', alpha=0.3)
    
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(corr_df_sorted['City'], rotation=45, ha='right')
    
    plt.title("Climate Change vs Agricultural Gap by City", fontsize=16, weight='normal')
    
    # Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=11)
    
    fig.tight_layout()
    
    filepath = os.path.join(output_dir, "03_DualAxis_Climate_Crop.png")
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved Plot 3: Dual axis comparison")

# =============================
# PLOT 4: Heatmap-style Matrix
# =============================
if len(corr_df) > 0:
    # Normalize values for color mapping
    from matplotlib.colors import TwoSlopeNorm
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 8))
    
    cities = corr_df['City'].values
    climate_vals = corr_df['Climate_Change_per_Decade_F'].values
    crop_vals = corr_df['Crop_Separation_Rate'].values
    
    y_pos = np.arange(len(cities))
    
    # Climate heatmap
    norm1 = TwoSlopeNorm(vmin=climate_vals.min(), vcenter=0, vmax=climate_vals.max())
    colors1 = plt.cm.RdYlBu_r(norm1(climate_vals))
    
    ax1.barh(y_pos, climate_vals, color=colors1, edgecolor='black', linewidth=1)
    ax1.set_yticks(y_pos)
    ax1.set_yticklabels(cities)
    ax1.set_xlabel('Temperature Change (°F/decade)', fontsize=12)
    ax1.set_title('Climate Warming Trend', fontsize=14, weight='normal')
    ax1.axvline(x=0, color='black', linewidth=2)
    ax1.grid(True, alpha=0.3, axis='x')
    
    # Crop separation heatmap
    norm2 = TwoSlopeNorm(vmin=crop_vals.min(), vcenter=0, vmax=crop_vals.max())
    colors2 = plt.cm.RdYlGn(norm2(crop_vals))
    
    ax2.barh(y_pos, crop_vals, color=colors2, edgecolor='black', linewidth=1)
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels([])
    ax2.set_xlabel('Crop Gap Rate (per year)', fontsize=12)
    ax2.set_title('Agricultural Productivity Gap', fontsize=14, weight='normal')
    ax2.axvline(x=0, color='black', linewidth=2)
    ax2.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    
    filepath = os.path.join(output_dir, "04_Heatmap_Comparison.png")
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved Plot 4: Heatmap comparison")

# =============================
# PLOT 5: Summary Statistics Table (FIXED)
# =============================
if len(corr_df) > 0:
    fig, ax = plt.subplots(figsize=(16, 10))
    ax.axis('tight')
    ax.axis('off')
    
    # Prepare summary data
    summary = corr_df.copy()
    summary['Climate_Change_per_Decade_F'] = summary['Climate_Change_per_Decade_F'].round(3)
    summary['Crop_Separation_Rate'] = summary['Crop_Separation_Rate'].round(4)
    summary['Final_Separation'] = summary['Final_Separation'].round(3)
    
    # Create table
    table_data = summary[['City', 'Country', 'Climate_Change_per_Decade_F', 'Crop_Separation_Rate', 
                          'Final_Separation', 'Years']].values
    
    col_labels = ['City', 'Country', 'Temp Change\n(°F/decade)', 'Crop Gap Rate\n(per year)', 
                  'Final Gap\n(growth factor)', 'Analysis\nPeriod']
    
    table = ax.table(cellText=table_data, colLabels=col_labels, 
                    cellLoc='center', loc='center',
                    colWidths=[0.15, 0.2, 0.15, 0.15, 0.15, 0.2])
    
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2.8)
    
    # Color code cells
    for i in range(1, len(table_data) + 1):
        # Climate change cell
        if float(table_data[i-1][2]) > 0:
            table[(i, 2)].set_facecolor('#ffcccc')
        
        # Crop separation cell
        if float(table_data[i-1][3]) < 0:
            table[(i, 3)].set_facecolor('#ffcccc')
        else:
            table[(i, 3)].set_facecolor('#ccffcc')
    
    # Style header
    for i, col in enumerate(col_labels):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    # Add title with MORE spacing from table
    fig.suptitle("Summary: Climate Change and Agricultural Productivity Correlation", 
                fontsize=16, weight='normal', y=0.98)
    
    # Add extra space at top by adjusting subplots
    plt.subplots_adjust(top=0.92, bottom=0.05, left=0.05, right=0.95)
    
    filepath = os.path.join(output_dir, "05_Summary_Table.png")
    plt.savefig(filepath, dpi=300, bbox_inches='tight', pad_inches=0.5)
    plt.close()
    print("✓ Saved Plot 5: Summary table")

# =============================
# Print Summary Statistics
# =============================
if len(corr_df) > 0:
    print("\n" + "="*60)
    print("CORRELATION ANALYSIS SUMMARY")
    print("="*60)
    
    if len(corr_df) >= 3:
        corr = np.corrcoef(corr_df['Climate_Change_per_Decade_F'], 
                          corr_df['Crop_Separation_Rate'])[0,1]
        print(f"\nPearson Correlation Coefficient: {corr:.3f}")
        
        if abs(corr) < 0.3:
            interpretation = "weak"
        elif abs(corr) < 0.7:
            interpretation = "moderate"
        else:
            interpretation = "strong"
        
        direction = "negative" if corr < 0 else "positive"
        print(f"Interpretation: {interpretation} {direction} correlation")
        
        if corr < 0:
            print("\nWarmer temperatures are associated with WIDENING productivity gaps")
            print("(Agriculture falling further behind expected growth)")
        else:
            print("\nWarmer temperatures are associated with NARROWING productivity gaps")
            print("(Agriculture keeping better pace with expected growth)")
    
    print("\n" + "="*60)
    print("ALL CORRELATION PLOTS SAVED!")
    print("="*60)
    print(f"Location: {output_dir}")
    print(f"Total plots: 5")

Saving correlation plots to: /Users/samadsulyman/Desktop/Data Science Project/Plots - Correlation

Loading Climate Data...

Loading Crop Production Data...

Calculating climate trends...

Calculating crop production separation rates...

Merging datasets...

Successfully matched 16 cities for correlation analysis
         City                          Country
        Accra                            Ghana
  Addis Abeba                         Ethiopia
  Addis Ababa                         Ethiopia
      Algiers                          Algeria
        Cairo                            Egypt
        Dakar                          Senegal
Dar Es Salaam      United Republic of Tanzania
 Johannesburg                     South Africa
      Kamalia                           Uganda
      Kampala                           Uganda
     Khartoum                            Sudan
     Kinshasa Democratic Republic of the Congo
        Lagos                          Nigeria
       Luanda               

In [260]:
# =============================
# Setup font to Times New Roman
# =============================
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif']
plt.rcParams['font.size'] = 11

# =============================
# Setup output directory
# =============================
output_dir = "/Users/samadsulyman/Desktop/Data Science Project/Plots - Wind_Precip_Climate"
os.makedirs(output_dir, exist_ok=True)
print(f"Saving wind/precipitation/climate correlation plots to: {output_dir}")

# =============================
# Load Dataset 4 (Weather Station Data)
# =============================
print("\n" + "="*60)
print("Loading Weather Station Data (Dataset 4)...")
print("="*60)

df4 = pd.read_csv("/Users/samadsulyman/Desktop/Data Science Project/Processed Data/ProcessedDataset4.csv")

# Convert date to datetime
df4["date"] = pd.to_datetime(df4["date"], errors="coerce")

# Convert numeric columns
numeric_cols = ['avg_temp_c', 'min_temp_c', 'max_temp_c', 'precipitation_mm', 
                'avg_wind_speed_kmh', 'peak_wind_gust_kmh']
for col in numeric_cols:
    df4[col] = pd.to_numeric(df4[col], errors='coerce')

print(f"Dataset shape: {df4.shape}")
print(f"Date range: {df4['date'].min()} to {df4['date'].max()}")
print(f"Cities: {df4['city_name'].nunique()}")
print(f"Unique cities: {sorted(df4['city_name'].unique())}")

# =============================
# Load Climate Data for Temperature Trends
# =============================
print("\n" + "="*60)
print("Loading Climate Data for Temperature Trends...")
print("="*60)

df1 = pd.read_csv("/Users/samadsulyman/Desktop/Data Science Project/Processed Data/ProcessedDataset1.csv")
df2 = pd.read_csv("/Users/samadsulyman/Desktop/Data Science Project/Processed Data/ProcessedDataset2.csv")

df1["date"] = pd.to_datetime(df1["date"], errors="coerce")
df2["date"] = pd.to_datetime(df2["date"], errors="coerce")
df1["Temp_F"] = df1["AverageTemperature"] * 9/5 + 32

# =============================
# Calculate Climate Trends (from previous analysis)
# =============================
def calculate_climate_trend(df, temp_col, start_year, end_year):
    """Calculate temperature change per decade for a city"""
    df = df[(df["date"].dt.year >= start_year) & (df["date"].dt.year <= end_year)].copy()
    
    if len(df) < 10:
        return {}
    
    # Monthly aggregation
    monthly = df.groupby([df["City"], df["date"].dt.to_period("M")])[temp_col].mean().reset_index()
    monthly["date"] = monthly["date"].dt.to_timestamp()
    
    results = {}
    for city in monthly["City"].unique():
        city_data = monthly[monthly["City"] == city].sort_values("date")
        
        # 6-month rolling average
        city_data["roll_6m"] = city_data[temp_col].rolling(window=6, center=False).mean()
        city_data = city_data.dropna(subset=["roll_6m"])
        
        if len(city_data) < 10:
            continue
        
        # Calculate linear trend
        x = city_data["date"].map(pd.Timestamp.toordinal).values
        y = city_data["roll_6m"].values
        
        coeffs = np.polyfit(x, y, 1)
        slope = coeffs[0]
        
        # Convert to change per decade
        years = (city_data["date"].iloc[-1] - city_data["date"].iloc[0]).days / 365.25
        temp_change = slope * (x[-1] - x[0])
        change_per_decade = (temp_change / years) * 10
        
        results[city] = change_per_decade
    
    return results

print("\nCalculating climate trends...")
climate_trends_1 = calculate_climate_trend(df1, "Temp_F", 1780, 2010)
climate_trends_2 = calculate_climate_trend(df2, "AvgTemperature", 1995, 2010)

# Combine climate trends
climate_trends = {**climate_trends_1, **climate_trends_2}

# =============================
# City Matching Dictionary
# =============================
city_matching = {
    # Dataset 4 cities -> Climate dataset cities
    'Abidjan': 'Abidjan',
    'Accra': 'Accra',
    'Addis Ababa': 'Addis Ababa',
    'Algiers': 'Algiers',
    'Cairo': 'Cairo',
    'Dakar': 'Dakar',
    'Dar Es Salaam': 'Dar Es Salaam',
    'Johannesburg': 'Johannesburg',
    'Kampala': 'Kampala',
    'Khartoum': 'Khartoum',
    'Kinshasa': 'Kinshasa',
    'Lagos': 'Lagos',
    'Luanda': 'Luanda',
    'Maputo': 'Maputo',
    'Nairobi': 'Nairobi'
}

# =============================
# Calculate Wind Speed and Precipitation Metrics by City
# =============================
print("\n" + "="*60)
print("Calculating wind speed and precipitation metrics...")
print("="*60)

def calculate_wind_precip_metrics(df):
    """Calculate average wind speed and precipitation by city"""
    results = {}
    
    for city in df['city_name'].unique():
        city_data = df[df['city_name'] == city].copy()
        
        # Remove null values
        city_data = city_data.dropna(subset=['avg_wind_speed_kmh', 'precipitation_mm'])
        
        if len(city_data) < 10:
            continue
        
        # Calculate yearly aggregates
        city_data['year'] = city_data['date'].dt.year
        yearly = city_data.groupby('year').agg({
            'avg_wind_speed_kmh': 'mean',
            'precipitation_mm': 'sum',
            'avg_temp_c': 'mean'
        }).reset_index()
        
        if len(yearly) < 3:
            continue
        
        # Calculate correlation between wind and precipitation
        if len(yearly) >= 3:
            wind_precip_corr = yearly['avg_wind_speed_kmh'].corr(yearly['precipitation_mm'])
        else:
            wind_precip_corr = None
        
        # Calculate trends over time
        x = yearly['year'].values
        
        # Wind speed trend
        if len(x) >= 3:
            wind_slope, _, wind_r, _, _ = stats.linregress(x, yearly['avg_wind_speed_kmh'].values)
            wind_change_per_decade = wind_slope * 10
        else:
            wind_change_per_decade = None
            wind_r = None
        
        # Precipitation trend
        if len(x) >= 3:
            precip_slope, _, precip_r, _, _ = stats.linregress(x, yearly['precipitation_mm'].values)
            precip_change_per_decade = precip_slope * 10
        else:
            precip_change_per_decade = None
            precip_r = None
        
        results[city] = {
            'avg_wind_speed': yearly['avg_wind_speed_kmh'].mean(),
            'total_precip': yearly['precipitation_mm'].mean(),
            'wind_precip_correlation': wind_precip_corr,
            'wind_trend_per_decade': wind_change_per_decade,
            'precip_trend_per_decade': precip_change_per_decade,
            'wind_trend_r': wind_r,
            'precip_trend_r': precip_r,
            'years_analyzed': f"{yearly['year'].min()}-{yearly['year'].max()}",
            'data_points': len(yearly)
        }
    
    return results

wind_precip_metrics = calculate_wind_precip_metrics(df4)

print(f"Calculated metrics for {len(wind_precip_metrics)} cities")

# =============================
# Merge All Data
# =============================
print("\n" + "="*60)
print("Merging wind/precipitation data with climate trends...")
print("="*60)

combined_data = []
for city_d4, city_climate in city_matching.items():
    if city_d4 in wind_precip_metrics and city_climate in climate_trends:
        metrics = wind_precip_metrics[city_d4]
        combined_data.append({
            'City': city_d4,
            'Climate_Change_per_Decade_F': climate_trends[city_climate],
            'Avg_Wind_Speed_kmh': metrics['avg_wind_speed'],
            'Avg_Annual_Precip_mm': metrics['total_precip'],
            'Wind_Precip_Correlation': metrics['wind_precip_correlation'],
            'Wind_Trend_per_Decade_kmh': metrics['wind_trend_per_decade'],
            'Precip_Trend_per_Decade_mm': metrics['precip_trend_per_decade'],
            'Years': metrics['years_analyzed'],
            'Data_Points': metrics['data_points']
        })

combined_df = pd.DataFrame(combined_data)

if len(combined_df) == 0:
    print("ERROR: No matching cities found!")
    print(f"Dataset 4 cities: {list(wind_precip_metrics.keys())}")
    print(f"Climate cities: {list(climate_trends.keys())}")
else:
    print(f"\nSuccessfully matched {len(combined_df)} cities")
    print(combined_df[['City', 'Climate_Change_per_Decade_F', 'Wind_Precip_Correlation']].to_string(index=False))

# =============================
# Print Summary Statistics
# =============================
if len(combined_df) > 0:
    print("\n" + "="*60)
    print("Creating Plot 1: Enhanced Multi-metric Heatmap...")
    print("="*60)
    
    from matplotlib.colors import TwoSlopeNorm
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 12))
    
    # Sort by climate change
    sorted_df = combined_df.sort_values('Climate_Change_per_Decade_F', ascending=False)
    cities = sorted_df['City'].values
    y_pos = np.arange(len(cities))
    
    # Plot 1: Climate Change
    climate_vals = sorted_df['Climate_Change_per_Decade_F'].values
    norm1 = TwoSlopeNorm(vmin=climate_vals.min(), vcenter=0, vmax=climate_vals.max())
    colors1 = plt.cm.RdYlBu_r(norm1(climate_vals))
    
    bars1 = axes[0].barh(y_pos, climate_vals, color=colors1, edgecolor='black', linewidth=1.2, height=0.7)
    axes[0].set_yticks(y_pos)
    axes[0].set_yticklabels(cities, fontsize=12, weight='bold')
    axes[0].set_xlabel('°F/decade', fontsize=13, weight='bold')
    axes[0].set_title('Temperature\nChange', fontsize=14, weight='bold', pad=15)
    axes[0].axvline(x=0, color='black', linewidth=2.5)
    axes[0].grid(True, alpha=0.3, axis='x', linewidth=1)
    
    # Add value labels
    for i, (bar, val) in enumerate(zip(bars1, climate_vals)):
        axes[0].text(val, i, f' {val:.2f}', va='center', ha='left' if val >= 0 else 'right', 
                    fontsize=10, weight='bold')
    
    # Plot 2: Wind Speed Trend
    wind_vals = sorted_df['Wind_Trend_per_Decade_kmh'].values
    norm2 = TwoSlopeNorm(vmin=np.nanmin(wind_vals), vcenter=0, vmax=np.nanmax(wind_vals))
    colors2 = plt.cm.RdYlGn(norm2(wind_vals))
    
    bars2 = axes[1].barh(y_pos, wind_vals, color=colors2, edgecolor='black', linewidth=1.2, height=0.7)
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels([])
    axes[1].set_xlabel('km/h/decade', fontsize=13, weight='bold')
    axes[1].set_title('Wind Speed\nTrend', fontsize=14, weight='bold', pad=15)
    axes[1].axvline(x=0, color='black', linewidth=2.5)
    axes[1].grid(True, alpha=0.3, axis='x', linewidth=1)
    
    # Add value labels
    for i, (bar, val) in enumerate(zip(bars2, wind_vals)):
        if not np.isnan(val):
            axes[1].text(val, i, f' {val:.1f}', va='center', ha='left' if val >= 0 else 'right',
                        fontsize=10, weight='bold')
    
    # Plot 3: Precipitation Trend
    precip_vals = sorted_df['Precip_Trend_per_Decade_mm'].values
    norm3 = TwoSlopeNorm(vmin=np.nanmin(precip_vals), vcenter=0, vmax=np.nanmax(precip_vals))
    colors3 = plt.cm.BrBG(norm3(precip_vals))
    
    bars3 = axes[2].barh(y_pos, precip_vals, color=colors3, edgecolor='black', linewidth=1.2, height=0.7)
    axes[2].set_yticks(y_pos)
    axes[2].set_yticklabels([])
    axes[2].set_xlabel('mm/decade', fontsize=13, weight='bold')
    axes[2].set_title('Precipitation\nTrend', fontsize=14, weight='bold', pad=15)
    axes[2].axvline(x=0, color='black', linewidth=2.5)
    axes[2].grid(True, alpha=0.3, axis='x', linewidth=1)
    
    # Add value labels
    for i, (bar, val) in enumerate(zip(bars3, precip_vals)):
        if not np.isnan(val):
            axes[2].text(val, i, f' {val:.0f}', va='center', ha='left' if val >= 0 else 'right',
                        fontsize=10, weight='bold')
    
    # Plot 4: Wind-Precipitation Correlation
    corr_vals = sorted_df['Wind_Precip_Correlation'].values
    norm4 = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
    colors4 = plt.cm.PiYG(norm4(corr_vals))
    
    bars4 = axes[3].barh(y_pos, corr_vals, color=colors4, edgecolor='black', linewidth=1.2, height=0.7)
    axes[3].set_yticks(y_pos)
    axes[3].set_yticklabels([])
    axes[3].set_xlabel('Correlation', fontsize=13, weight='bold')
    axes[3].set_title('Wind-Precip\nCorrelation', fontsize=14, weight='bold', pad=15)
    axes[3].axvline(x=0, color='black', linewidth=2.5)
    axes[3].grid(True, alpha=0.3, axis='x', linewidth=1)
    axes[3].set_xlim(-1, 1)
    
    # Add value labels
    for i, (bar, val) in enumerate(zip(bars4, corr_vals)):
        if not np.isnan(val):
            axes[3].text(val, i, f' {val:.2f}', va='center', ha='left' if val >= 0 else 'right',
                        fontsize=10, weight='bold')
    
    fig.suptitle('Climate Change, Wind, and Precipitation Patterns Across African Cities', 
                fontsize=18, weight='bold', y=0.98)
    
    plt.subplots_adjust(top=0.94, bottom=0.06, left=0.08, right=0.98, wspace=0.25)
    
    filepath = os.path.join(output_dir, "01_Multi_Metric_Heatmap.png")
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved Plot 1: Enhanced Heatmap")

# =============================
# PLOT 2: Correlation Matrix Table
# =============================
if len(combined_df) > 0:
    print("\n" + "="*60)
    print("Creating Plot 2: Correlation Matrix Table...")
    print("="*60)
    
    # Prepare correlation matrix
    metrics_for_corr = combined_df[['Climate_Change_per_Decade_F', 'Wind_Trend_per_Decade_kmh', 
                                     'Precip_Trend_per_Decade_mm', 'Wind_Precip_Correlation']].copy()
    
    # Calculate correlation matrix
    corr_matrix = metrics_for_corr.corr()
    
    # Create figure
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.axis('tight')
    ax.axis('off')
    
    # Prepare data for table
    metric_names = ['Temperature\nChange\n(°F/decade)', 
                    'Wind Speed\nTrend\n(km/h/decade)', 
                    'Precipitation\nTrend\n(mm/decade)',
                    'Wind-Precip\nCorrelation']
    
    # Create table data with correlations
    table_data = []
    for i, row_name in enumerate(metric_names):
        row = [row_name]
        for j in range(len(metric_names)):
            corr_val = corr_matrix.iloc[i, j]
            if i == j:
                row.append('1.000')
            else:
                row.append(f'{corr_val:.3f}')
        table_data.append(row)
    
    # Column labels
    col_labels = ['Metric'] + metric_names
    
    # Create table
    table = ax.table(cellText=table_data, colLabels=col_labels,
                    cellLoc='center', loc='center',
                    colWidths=[0.22, 0.19, 0.19, 0.19, 0.19])
    
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1, 4)
    
    # Color code cells based on correlation strength
    from matplotlib.colors import LinearSegmentedColormap
    
    # Create custom colormap (red for negative, white for zero, blue for positive)
    colors_map = [(0.8, 0.2, 0.2), (1, 1, 1), (0.2, 0.4, 0.8)]
    n_bins = 100
    cmap = LinearSegmentedColormap.from_list('correlation', colors_map, N=n_bins)
    
    for i in range(1, len(table_data) + 1):
        for j in range(1, len(col_labels)):
            cell_val = float(table_data[i-1][j])
            
            # Color based on correlation strength
            color = cmap((cell_val + 1) / 2)  # Normalize -1 to 1 -> 0 to 1
            table[(i, j)].set_facecolor(color)
            
            # Make diagonal cells bold
            if i == j:
                table[(i, j)].set_text_props(weight='bold', fontsize=13)
            
            # Strong correlations in bold
            if abs(cell_val) > 0.5 and i != j:
                table[(i, j)].set_text_props(weight='bold', fontsize=13)
    
    # Style header row
    for j in range(len(col_labels)):
        table[(0, j)].set_facecolor('#2C5F7C')
        table[(0, j)].set_text_props(weight='bold', color='white', fontsize=12)
        table[(0, j)].set_height(0.12)
    
    # Style first column
    for i in range(1, len(table_data) + 1):
        table[(i, 0)].set_facecolor('#E8F4F8')
        table[(i, 0)].set_text_props(weight='bold', fontsize=11, ha='left')
    
    # Add title
    fig.suptitle('Correlation Matrix: Climate, Wind, and Precipitation Metrics', 
                fontsize=18, weight='bold', y=0.96)
    
    # Add interpretation text
    interpretation_text = (
        "Interpretation Guide:\n"
        "• Values close to +1.0: Strong positive correlation (as one increases, the other increases)\n"
        "• Values close to -1.0: Strong negative correlation (as one increases, the other decreases)\n"
        "• Values close to 0.0: Weak or no correlation\n"
        "• Diagonal (1.000): Each metric perfectly correlates with itself"
    )
    
    fig.text(0.5, 0.08, interpretation_text, ha='center', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3, pad=15),
            multialignment='left')
    
    # Add color legend
    legend_text = "Color Scale:   Red = Negative Correlation   |   White = No Correlation   |   Blue = Positive Correlation"
    fig.text(0.5, 0.02, legend_text, ha='center', fontsize=11, style='italic',
            weight='bold')
    
    plt.subplots_adjust(top=0.92, bottom=0.20, left=0.05, right=0.95)
    
    filepath = os.path.join(output_dir, "02_Correlation_Matrix_Table.png")
    plt.savefig(filepath, dpi=300, bbox_inches='tight', pad_inches=0.5)
    plt.close()
    print("✓ Saved Plot 2: Correlation Matrix Table")
if len(combined_df) > 0:
    print("\n" + "="*60)
    print("SUMMARY: Wind, Precipitation & Climate Correlations")
    print("="*60)
    
    # Display full summary table
    summary_display = combined_df.copy()
    summary_display['Climate_Change_per_Decade_F'] = summary_display['Climate_Change_per_Decade_F'].round(3)
    summary_display['Wind_Trend_per_Decade_kmh'] = summary_display['Wind_Trend_per_Decade_kmh'].round(3)
    summary_display['Precip_Trend_per_Decade_mm'] = summary_display['Precip_Trend_per_Decade_mm'].round(1)
    summary_display['Wind_Precip_Correlation'] = summary_display['Wind_Precip_Correlation'].round(3)
    
    print("\n" + summary_display.to_string(index=False))
    
    # Calculate overall correlations
    print("\n" + "="*60)
    print("OVERALL CORRELATIONS:")
    print("="*60)
    
    valid_climate_wind = combined_df.dropna(subset=['Climate_Change_per_Decade_F', 'Wind_Trend_per_Decade_kmh'])
    if len(valid_climate_wind) >= 3:
        corr_cw = valid_climate_wind['Climate_Change_per_Decade_F'].corr(valid_climate_wind['Wind_Trend_per_Decade_kmh'])
        print(f"Climate Change vs Wind Speed Trend: {corr_cw:.3f}")
    
    valid_climate_precip = combined_df.dropna(subset=['Climate_Change_per_Decade_F', 'Precip_Trend_per_Decade_mm'])
    if len(valid_climate_precip) >= 3:
        corr_cp = valid_climate_precip['Climate_Change_per_Decade_F'].corr(valid_climate_precip['Precip_Trend_per_Decade_mm'])
        print(f"Climate Change vs Precipitation Trend: {corr_cp:.3f}")
    
    valid_wind_precip = combined_df.dropna(subset=['Wind_Trend_per_Decade_kmh', 'Precip_Trend_per_Decade_mm'])
    if len(valid_wind_precip) >= 3:
        corr_wp = valid_wind_precip['Wind_Trend_per_Decade_kmh'].corr(valid_wind_precip['Precip_Trend_per_Decade_mm'])
        print(f"Wind Speed Trend vs Precipitation Trend: {corr_wp:.3f}")
    
    valid_climate_corr = combined_df.dropna(subset=['Climate_Change_per_Decade_F', 'Wind_Precip_Correlation'])
    if len(valid_climate_corr) >= 3:
        corr_cc = valid_climate_corr['Climate_Change_per_Decade_F'].corr(valid_climate_corr['Wind_Precip_Correlation'])
        print(f"Climate Change vs Wind-Precip Correlation: {corr_cc:.3f}")
    
    print("\n" + "="*60)
    print("INTERPRETATION:")
    print("="*60)
    print("Positive wind trend: Wind speeds INCREASING over time")
    print("Negative wind trend: Wind speeds DECREASING over time")
    print("Positive precip trend: Precipitation INCREASING over time")
    print("Negative precip trend: Precipitation DECREASING over time")
    print("Positive wind-precip correlation: Windier periods have MORE rain")
    print("Negative wind-precip correlation: Windier periods have LESS rain")
    
    print("\n" + "="*60)
    print("ALL WIND/PRECIPITATION/CLIMATE PLOTS SAVED!")
    print("="*60)
    print(f"Location: {output_dir}")
    print(f"Total plots: 2")
    print(f"  - Plot 1: Enhanced Multi-Metric Heatmap")
    print(f"  - Plot 2: Correlation Matrix Table")
    print(f"Cities analyzed: {len(combined_df)}")

Saving wind/precipitation/climate correlation plots to: /Users/samadsulyman/Desktop/Data Science Project/Plots - Wind_Precip_Climate

Loading Weather Station Data (Dataset 4)...


/var/folders/26/2f2vy6w539b6382kp83vyy_w0000gn/T/ipykernel_30793/3375731183.py:22: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df4 = pd.read_csv("/Users/samadsulyman/Desktop/Data Science Project/Processed Data/ProcessedDataset4.csv")


Dataset shape: (158144, 16)
Date range: 1942-09-07 00:00:00 to 2023-09-05 00:00:00
Cities: 11
Unique cities: ['Abidjan', 'Accra', 'Addis Ababa', 'Algiers', 'Dakar', 'Dar es Salaam', 'Khartoum', 'Lagos', 'Luanda', 'Maputo', 'Nairobi']

Loading Climate Data for Temperature Trends...

Calculating climate trends...

Calculating wind speed and precipitation metrics...
Calculated metrics for 11 cities

Merging wind/precipitation data with climate trends...

Successfully matched 10 cities
       City  Climate_Change_per_Decade_F  Wind_Precip_Correlation
    Abidjan                    -1.192565                 0.104611
      Accra                     0.175123                -0.395133
Addis Ababa                     0.755291                 0.203477
    Algiers                     0.098007                -0.453999
      Dakar                     0.648831                -0.342665
   Khartoum                     0.136877                 0.476479
      Lagos                    -0.592480           

In [280]:
# ============================= 
# Climate Hotspot Map Fix (2000-2020) - NO OVERLAP
# ============================= 
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import scipy.stats as stats
import os

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif']
plt.rcParams['font.size'] = 11

output_dir = "/Users/samadsulyman/Desktop/Data Science Project/Plots - Heat_Vulnerability"
os.makedirs(output_dir, exist_ok=True)
print(f"Fixing Climate Hotspot Map in: {output_dir}")

# ============================= 
# Load Datasets
# ============================= 
print("\n" + "="*60)
print("Loading datasets...")
print("="*60)

df1 = pd.read_csv("/Users/samadsulyman/Desktop/Data Science Project/Processed Data/ProcessedDataset1.csv")
df2 = pd.read_csv("/Users/samadsulyman/Desktop/Data Science Project/Processed Data/ProcessedDataset2.csv")

df1["date"] = pd.to_datetime(df1["date"], errors="coerce")
df2["date"] = pd.to_datetime(df2["date"], errors="coerce")
df1["Temp_C"] = df1["AverageTemperature"]
df2["Temp_C"] = df2["AvgTemperature"]

print("Datasets loaded successfully")

# ============================= 
# City Coordinates and Country Info
# ============================= 
city_coords = {
    'Abidjan': (5.3600, -4.0083, "Côte d'Ivoire"),
    'Accra': (5.6037, -0.1870, "Ghana"),
    'Addis Ababa': (9.0320, 38.7469, "Ethiopia"),
    'Addis Abeba': (9.0320, 38.7469, "Ethiopia"),
    'Algiers': (36.7538, 3.0588, "Algeria"),
    'Cairo': (30.0444, 31.2357, "Egypt"),
    'Dakar': (14.7167, -17.4677, "Senegal"),
    'Dar Es Salaam': (-6.7924, 39.2083, "Tanzania"),
    'Johannesburg': (-26.2041, 28.0473, "South Africa"),
    'Kampala': (0.3476, 32.5825, "Uganda"),
    'Khartoum': (15.5007, 32.5599, "Sudan"),
    'Kinshasa': (-4.4419, 15.2663, "DR Congo"),
    'Lagos': (6.5244, 3.3792, "Nigeria"),
    'Luanda': (-8.8383, 13.2344, "Angola"),
    'Maputo': (-25.9692, 32.5732, "Mozambique"),
    'Nairobi': (-1.2864, 36.8172, "Kenya")
}

# ============================= 
# Calculate Heat Vulnerability Metrics
# ============================= 
def calculate_heat_metrics(df, period_start, period_end):
    """Calculate heat vulnerability metrics for a time period"""
    period_data = df[(df['date'].dt.year >= period_start) & 
                     (df['date'].dt.year <= period_end)].copy()
    
    results = []
    
    for city in period_data['City'].unique():
        city_data = period_data[period_data['City'] == city].copy()
        if len(city_data) < 10:
            continue
        
        city_data['year'] = city_data['date'].dt.year
        city_data['month'] = city_data['date'].dt.month
        yearly = city_data.groupby('year')['Temp_C'].agg(['mean', 'max', 'std']).reset_index()
        
        if len(yearly) < 3:
            continue
        
        avg_temp = yearly['mean'].mean()
        max_temp = yearly['max'].max()
        slope, _, r_val, _, _ = stats.linregress(yearly['year'], yearly['mean'])
        temp_trend = slope * 10
        temp_variability = yearly['std'].mean()
        
        temp_95 = city_data['Temp_C'].quantile(0.95)
        extreme_heat_pct = (city_data['Temp_C'] > temp_95).sum() / len(city_data) * 100
        
        lat = city_coords.get(city, (0, 0, ""))[0]
        summer_months = [6, 7, 8] if lat >= 0 else [12, 1, 2]
        summer_data = city_data[city_data['month'].isin(summer_months)]
        summer_avg = summer_data['Temp_C'].mean() if len(summer_data) > 0 else avg_temp
        
        vulnerability_score = (
            (avg_temp / 35.0) * 20 +
            (max(0, temp_trend) * 5) +
            (temp_variability * 2) +
            (extreme_heat_pct / 5.0) * 20 +
            (summer_avg / 40.0) * 20
        )
        vulnerability_score = min(100, vulnerability_score)
        
        results.append({
            'City': city,
            'Period': f"{period_start}-{period_end}",
            'Avg_Temp_C': avg_temp,
            'Max_Temp_C': max_temp,
            'Temp_Trend_per_Decade': temp_trend,
            'Temp_Variability': temp_variability,
            'Extreme_Heat_Pct': extreme_heat_pct,
            'Summer_Avg_C': summer_avg,
            'Vulnerability_Score': vulnerability_score
        })
    
    return pd.DataFrame(results)

# ============================= 
# Calculate for 2000-2020 Period
# ============================= 
print("\n" + "="*60)
print("Calculating heat vulnerability for 2000-2020...")
print("="*60)

latest_data = calculate_heat_metrics(df1, 2000, 2020)
if len(latest_data) == 0:
    latest_data = calculate_heat_metrics(df2, 2000, 2020)

latest_data['Latitude'] = latest_data['City'].map(
    lambda x: city_coords.get(x, (0, 0, ""))[0])
latest_data['Longitude'] = latest_data['City'].map(
    lambda x: city_coords.get(x, (0, 0, ""))[1])

# Define hotspot criteria
latest_data['Is_Hotspot'] = (
    (latest_data['Vulnerability_Score'] > 60) |
    (latest_data['Temp_Trend_per_Decade'] > 0.15) |
    (latest_data['Extreme_Heat_Pct'] > 6)
)

latest_data['Hotspot_Level'] = 'Low Risk'
latest_data.loc[latest_data['Vulnerability_Score'] > 50, 'Hotspot_Level'] = 'Moderate Risk'
latest_data.loc[latest_data['Vulnerability_Score'] > 65, 'Hotspot_Level'] = 'High Risk'
latest_data.loc[latest_data['Vulnerability_Score'] > 80, 'Hotspot_Level'] = 'Extreme Risk'

print(f"Found data for {len(latest_data)} cities")

# ============================= 
# PLOT 3: Climate Hotspot Identification - FIXED NO OVERLAP
# ============================= 
print("\n" + "="*60)
print("Creating climate hotspot identification map (FIXED)...")
print("="*60)

fig, ax = plt.subplots(figsize=(16, 14))
ax.set_xlim(-20, 55)
ax.set_ylim(-35, 40)
ax.set_aspect('equal')
ax.grid(True, alpha=0.2, linestyle='--', linewidth=0.5)

# Color mapping for risk levels
risk_colors = {
    'Low Risk': '#2ECC40',
    'Moderate Risk': '#FFDC00',
    'High Risk': '#FF851B',
    'Extreme Risk': '#FF4136'
}

latest_data['Color'] = latest_data['Hotspot_Level'].map(risk_colors)

# Plot cities by risk level
for risk_level in ['Low Risk', 'Moderate Risk', 'High Risk', 'Extreme Risk']:
    risk_data = latest_data[latest_data['Hotspot_Level'] == risk_level]
    if len(risk_data) > 0:
        ax.scatter(risk_data['Longitude'], risk_data['Latitude'], 
                  s=risk_data['Vulnerability_Score'] * 10,
                  c=risk_data['Color'], 
                  alpha=0.75, 
                  edgecolor='black', 
                  linewidth=2.5,
                  label=risk_level, 
                  zorder=5)

# FIXED: Add labels with NO OVERLAP using adjustText or manual spacing
texts = []
for _, row in latest_data.iterrows():
    label_text = (f"{row['City']}\n"
                 f"Score: {row['Vulnerability_Score']:.0f}\n"
                 f"Trend: +{row['Temp_Trend_per_Decade']:.2f}°C/dec")
    
    text = ax.text(row['Longitude'], row['Latitude'], label_text,
                  fontsize=9, weight='bold', ha='center', va='center',
                  bbox=dict(boxstyle='round,pad=0.5', facecolor='white', 
                           alpha=0.9, edgecolor=row['Color'], linewidth=3))
    texts.append(text)

# Try to use adjustText for automatic overlap prevention
try:
    from adjustText import adjust_text
    adjust_text(texts, ax=ax,
               arrowprops=dict(arrowstyle='->', color='gray', lw=1),
               expand_points=(2, 2),
               expand_text=(1.2, 1.2),
               force_text=(0.5, 0.5),
               force_points=(0.3, 0.3))
    print("  Using adjustText for automatic label positioning")
except ImportError:
    print("  Using manual offset patterns (install adjustText for better results)")
    # ALTERNATIVE: Strategic manual offsets based on geographic location
    # Sort by latitude to handle north-south spacing
    sorted_data = latest_data.sort_values('Latitude', ascending=False).reset_index(drop=True)
    
    # Define smart offsets based on city clusters
    manual_offsets = {
        'Algiers': (0, 2.5),
        'Cairo': (3, 1.5),
        'Khartoum': (3, -1),
        'Dakar': (-3, 2),
        'Abidjan': (-2, -2),
        'Lagos': (2, -2),
        'Accra': (-3, 0),
        'Addis Ababa': (3, 1),
        'Addis Abeba': (3, 1),
        'Kampala': (-2, 2),
        'Nairobi': (2, -2),
        'Kinshasa': (-3, -1),
        'Dar Es Salaam': (3, 0),
        'Luanda': (-3, 1),
        'Johannesburg': (0, -2.5),
        'Maputo': (3, -1),
        'Kamalia': (-2, 2)
    }
    
    for text, (_, row) in zip(texts, latest_data.iterrows()):
        city = row['City']
        if city in manual_offsets:
            dx, dy = manual_offsets[city]
            text.set_position((row['Longitude'] + dx, row['Latitude'] + dy))

ax.set_xlabel('Longitude', fontsize=13, weight='bold')
ax.set_ylabel('Latitude', fontsize=13, weight='bold')
ax.set_title('Climate Hotspot Identification Map (2000-2020)\nVulnerability Assessment', 
            fontsize=18, weight='bold', pad=20)

ax.legend(loc='lower left', fontsize=12, frameon=True, fancybox=False, 
         edgecolor='black', framealpha=0.9, title='Risk Level', title_fontsize=13)

# Add hotspot criteria box
criteria_text = ("Hotspot Criteria:\n"
                "• Vulnerability Score > 60\n"
                "• Warming Trend > 0.15°C/decade\n"
                "• Extreme Heat Events > 6%")
ax.text(0.02, 0.98, criteria_text, transform=ax.transAxes, 
       fontsize=11, verticalalignment='top',
       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8, pad=10))

plt.tight_layout()
filepath = os.path.join(output_dir, "03_Climate_Hotspot_Map.png")
plt.savefig(filepath, dpi=300, bbox_inches='tight')
plt.close()

print("✓ Saved fixed climate hotspot map (no overlap)")
print("\n" + "="*60)
print("CLIMATE HOTSPOT MAP UPDATED SUCCESSFULLY!")
print("="*60)
print(f"Location: {filepath}")
print(f"Identified {latest_data['Is_Hotspot'].sum()} climate hotspots")

Fixing Climate Hotspot Map in: /Users/samadsulyman/Desktop/Data Science Project/Plots - Heat_Vulnerability

Loading datasets...
Datasets loaded successfully

Calculating heat vulnerability for 2000-2020...
Found data for 16 cities

Creating climate hotspot identification map (FIXED)...
  Using adjustText for automatic label positioning
✓ Saved fixed climate hotspot map (no overlap)

CLIMATE HOTSPOT MAP UPDATED SUCCESSFULLY!
Location: /Users/samadsulyman/Desktop/Data Science Project/Plots - Heat_Vulnerability/03_Climate_Hotspot_Map.png
Identified 11 climate hotspots
